In [7]:
%pip install google-genai pydantic textstat pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\arsh\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [16]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\arsh\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [17]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: C:\Users\arsh\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


# FaithfulMed — single-agent baseline (Gemini)

**Owner:** Arshiya Salehi  
**Purpose:** September milestone **single-Gemini baseline** — one API call that internally performs Extractor → Simplifier → Verifier → Refiner → Readability, returning structured JSON comparable to the future [adk_pipeline.py](adk_pipeline.py) multi-agent design.

- **EDA / MedAESQA:** see [arshiya.ipynb](arshiya.ipynb)
- **This notebook:** simplification baseline only (not MedAESQA Q&A eval)
- **October:** split stages into separate ADK agents and compare against runs saved here


In [20]:
# Google AI Studio API key — https://aistudio.google.com/apikey
# Do NOT commit real keys. Prefer environment variables or .env files.

import os
from dotenv import load_dotenv

load_dotenv() # This loads variables from your .env file

api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not api_key:
    print("GEMINI_API_KEY is not set. Please set it in your .env file or as an environment variable.")
else:
    print("GEMINI_API_KEY is loaded and will be used.")

GEMINI_API_KEY is loaded and will be used.


In [21]:
# Optional: run once if packages missing
# %pip install google-genai pydantic textstat pandas

from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from typing import Literal

import pandas as pd
from google import genai
from google.genai import errors as genai_errors
from google.genai import types
from pydantic import BaseModel, Field

# Repo paths
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
NOTEBOOKS = ROOT / "notebooks"
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))

from eval_harness import meets_readability_target, readability_scores

MODEL = "gemini-3.8-flash"  # swap for leaderboard experiments


In [22]:
def get_gemini_client() -> genai.Client:
    api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        api_key = getpass("Enter GEMINI_API_KEY from AI Studio (hidden): ").strip()
    if not api_key:
        raise ValueError(
            "Set GEMINI_API_KEY (https://aistudio.google.com/apikey) or enter it when prompted."
        )
    
    return genai.Client(api_key=api_key)


client = get_gemini_client()
print("Gemini client ready.")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Gemini client ready.


In [23]:
AtomType = Literal[
    "diagnosis", "medication", "lab_value", "procedure", "instruction", "other"
]


class ClinicalAtom(BaseModel):
    text: str = Field(description="One factual clinical atom from the SOURCE only.")
    type: AtomType


class Verdict(BaseModel):
    faithful: bool = Field(description="True if draft contains only SOURCE-supported facts.")
    unsupported_claims: list[str] = Field(default_factory=list)
    omissions: list[str] = Field(default_factory=list)
    reading_level_ok: bool = Field(
        description="True if draft is roughly 8th-grade reading level or below."
    )


class FaithfulMedOutput(BaseModel):
    """Single-call output mirroring the five-agent pipeline keys in adk_pipeline.py."""

    atoms: list[ClinicalAtom]
    draft: str = Field(description="First plain-language rewrite of the SOURCE.")
    verdict: Verdict
    final: str = Field(description="Patient-facing explanation after refine + readability pass.")
    reasoning_trace: str = Field(
        default="",
        description="Brief summary of internal Extractor/Simplifier/Verifier/Refiner/Readability steps.",
    )


SYSTEM_INSTRUCTION = """You are FaithfulMed, a single agent that must complete five internal steps before responding:

1. EXTRACTOR: List every clinically meaningful fact in SOURCE as atoms {text, type}.
   Types: diagnosis, medication, lab_value, procedure, instruction, other.
   Do not invent facts not present in SOURCE.

2. SIMPLIFIER: Write a plain-language draft for a patient using ONLY those atoms.

3. VERIFIER: Compare draft to atoms. Set faithful=false if any unsupported claim or omission.
   List unsupported_claims and omissions explicitly.

4. REFINER: If not faithful, fix draft (remove unsupported claims, restore omissions).
   If faithful, keep draft content unchanged.

5. READABILITY: Polish to Flesch-Kincaid grade 8 or below while preserving every fact.
   Put the polished text in `final`.

Rules:
- Do not add diagnoses, treatments, or advice not in SOURCE.
- Preserve uncertainty and negation from SOURCE.
- `final` is the only patient-facing output; keep it clear and respectful.
- Fill `reasoning_trace` with a short (≤5 sentences) note on what you checked.
"""


def run_single_agent(
    source: str,
    *,
    client: genai.Client | None = None,
    model: str = MODEL,
    max_retries: int = 1,
) -> tuple[FaithfulMedOutput, float]:
    """One Gemini call; returns parsed output and latency in seconds."""
    client = client or get_gemini_client()
    user_content = f"SOURCE:\n{source.strip()}\n\nReturn the JSON object for all pipeline stages."

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        response_mime_type="application/json",
        response_schema=FaithfulMedOutput,
        temperature=0.2,
    )

    last_err: Exception | None = None
    for attempt in range(max_retries + 1):
        t0 = time.perf_counter()
        try:
            response = client.models.generate_content(
                model=model,
                contents=user_content,
                config=config,
            )
            latency = time.perf_counter() - t0
            if response.parsed is not None:
                return response.parsed, latency
            # fallback parse
            text = response.text or ""
            data = json.loads(text)
            return FaithfulMedOutput.model_validate(data), latency
        except Exception as exc:
            last_err = exc
            #msg = str(exc)
            if attempt >= max_retries:
                break
    raise RuntimeError(f"Single-agent call failed after retries: {last_err}")


In [25]:
print("Testing model:", MODEL)

response = client.models.generate_content(
    model=MODEL,
    contents="Reply with only: OK",
)

print(response.text)

Testing model: gemini-3.8-flash


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [19]:
SAMPLE_COUGH = """The patient reports a dry cough for three days. No fever or
shortness of breath is reported. A viral upper respiratory infection is
suspected. The plan is rest, fluids, and follow-up if symptoms worsen or persist."""

SAMPLE_DISCHARGE = """DISCHARGE SUMMARY
Patient was admitted for acute decompensated heart failure. Echocardiogram showed
reduced ejection fraction of 35%. Discharged on furosemide 40 mg daily and
carvedilol 6.25 mg twice daily. Low-sodium diet and daily weights instructed.
Follow up with cardiology in 2 weeks. Return to ED for worsening shortness of
breath, chest pain, or weight gain greater than 3 pounds in one day."""

SAMPLES = [
    {"sample_id": "demo_cough", "source": SAMPLE_COUGH},
    {"sample_id": "demo_discharge_hf", "source": SAMPLE_DISCHARGE},
]

# Single demo run (first sample)
demo_source = SAMPLES[0]["source"]
#result, latency_s = run_single_agent(demo_source, client=client)
result, latency_s = run_single_agent(
    SAMPLES[0]["source"],
    client=client,
    model=MODEL,
)
print(result.final)
print(f"Model: {MODEL} | Latency: {latency_s:.2f}s")
print(f"Atoms extracted: {len(result.atoms)}")
print(f"Verdict faithful: {result.verdict.faithful}")
print("\n--- FINAL (patient-facing) ---\n")
print(result.final)


RuntimeError: Single-agent call failed after retries: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [16]:
from IPython.display import Markdown, display

scores = readability_scores(result.final)
print("Readability (final):", scores)
print("Meets <= 8th grade (FK):", meets_readability_target(result.final))

display(
    Markdown(
        "### Source vs final\n\n"
        f"**Source**\n\n{demo_source}\n\n---\n\n"
        f"**Final**\n\n{result.final}\n\n---\n\n"
        f"**Trace:** {result.reasoning_trace}"
    )
)

payload = json.dumps(result.model_dump(), indent=2)
print("\nStructured pipeline artifacts:")
print(payload if len(payload) <= 4000 else payload[:4000] + "\n... [truncated]")


Readability (final): {'flesch_kincaid_grade': 5.863636363636363, 'smog_index': 8.841846274778883, 'jargon_density': 0.18181818181818182, 'word_count': 44}
Meets <= 8th grade (FK): True


### Source vs final

**Source**

The patient reports a dry cough for three days. No fever or
shortness of breath is reported. A viral upper respiratory infection is
suspected. The plan is rest, fluids, and follow-up if symptoms worsen or persist.

---

**Final**

You reported having a dry cough for three days, with no fever and no trouble breathing. A viral upper respiratory infection is suspected. The plan is to rest and drink fluids. Please follow up if your symptoms get worse or do not go away.

---

**Trace:** Extracted all clinical findings, negations, the suspected diagnosis, and care instructions. Verified that the draft contains no unmentioned advice or diagnoses. Preserved the uncertainty regarding the viral infection. Confirmed reading level is well below grade 8.


Structured pipeline artifacts:
{
  "atoms": [
    {
      "text": "Dry cough for three days",
      "type": "other"
    },
    {
      "text": "No fever reported",
      "type": "other"
    },
    {
      "text": "No shortness of breath reported",
      "type": "other"
    },
    {
      "text": "Suspected viral upper respiratory infection",
      "type": "diagnosis"
    },
    {
      "text": "Plan includes rest",
      "type": "instruction"
    },
    {
      "text": "Plan includes fluids",
      "type": "instruction"
    },
    {
      "text": "Follow-up if symptoms worsen or persist",
      "type": "instruction"
    }
  ],
  "draft": "You have had a dry cough for three days, with no fever or trouble breathing. Your provider suspects a viral upper respiratory infection. The plan is to get rest, drink fluids, and follow up if your symptoms get worse or do not improve.",
  "verdict": {
    "faithful": true,
    "unsupported_claims": [],
    "omissions": [],
    "reading_level_ok": tr

In [17]:
RUN_BATCH = True  # set False to skip API calls during quick re-runs

rows = []
if RUN_BATCH:
    for item in SAMPLES:
        out, lat = run_single_agent(item["source"], client=client)
        read = readability_scores(out.final)
        rows.append(
            {
                "run_at_utc": datetime.now(timezone.utc).isoformat(),
                "model": MODEL,
                "sample_id": item["sample_id"],
                "latency_s": round(lat, 3),
                "n_atoms": len(out.atoms),
                "verdict_faithful": out.verdict.faithful,
                "n_unsupported": len(out.verdict.unsupported_claims),
                "n_omissions": len(out.verdict.omissions),
                "flesch_kincaid_grade": read["flesch_kincaid_grade"],
                "meets_8th_grade": meets_readability_target(out.final),
                "word_count": read["word_count"],
                "final_preview": out.final[:200].replace("\n", " "),
            }
        )

    df_runs = pd.DataFrame(rows)
    display(df_runs)

    out_csv = NOTEBOOKS / "single_agent_baseline_runs.csv"
    df_runs.to_csv(out_csv, index=False)
    print(f"Saved {out_csv}")
else:
    print("Batch skipped (RUN_BATCH=False).")


RuntimeError: Single-agent call failed after retries: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## Next steps (team)

1. **Curate ~50 examples** from MTSamples / hand-picked discharge snippets; add a loader cell (do not commit raw MTSamples to git).
2. **Freeze** the gold set and append each experiment as a row in `single_agent_baseline_runs.csv`.
3. **October:** implement [adk_pipeline.py](adk_pipeline.py) with five agents; compare faithfulness/readability/latency to this baseline.
4. **MedAESQA:** use only for **Verifier calibration** (human accuracy labels), not as the primary simplification benchmark — see [arshiya.ipynb](arshiya.ipynb).
5. **Simplifier RAG:** add MedlinePlus glossary index when moving beyond prompt-only simplification.
